### Formulation

The propagator is
$$
\begin{gather*}
    K(q_f, q_i, t)=\langle q_f|e^{-i\hat{H}t/\hbar}|q_i\rangle=\int\mathcal{D}q~e^{iS[q]/\hbar}\\
    S[q]=\int dt~\bigg(\frac{1}{2}m\dot{q}^2-V(q)\bigg)
\end{gather*}
$$
Using the Wick rotation, $t=-i\tau$,
$$
\begin{gather*}
    \dot{q}^2=\bigg(\frac{dq}{dt}\bigg)^2=\bigg(\frac{dq}{d\tau}\frac{1}{-i}\bigg)^2=-\bigg(\frac{dq}{d\tau}\bigg)^2\\
    \frac{i}{\hbar}S[q]=\frac{i}{\hbar}\int -id\tau~\bigg(-\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2-V(q)\bigg)=\frac{-1}{\hbar}\int d\tau~\bigg(\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2+V(q)\bigg)=\frac{-1}{\hbar}S_E[q]\\
    \therefore K_E(q_f, q_i, \tau)=\langle q_f|e^{-\hat{H}\tau/\hbar}|q_i\rangle=\int\mathcal{D}q~e^{-S_E[q]/\hbar}
\end{gather*}
$$
with the Euclidean action $S_E$,
$$
\begin{gather*}
    S_E[q]=\int_0^\tau d\tau~\bigg(\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2+V(q(\tau))\bigg)
\end{gather*}
$$
This Formulation implies connection to statistical physics. The partition function is
$$
\begin{gather*}
    Z=\mathrm{tr}(e^{-\beta\hat{H}})=\int dq~\langle q|e^{-\beta\hat{H}}|q\rangle
\end{gather*}
$$
If we replace $\tau=\hbar\beta$,
$$
\begin{gather*}
    Z=\int dq~\langle q|e^{-\beta\hat{H}}|q\rangle=\int dq~K_E(q, q, \hbar\beta)=\int\mathcal{D}q~e^{-S_E[q]/\hbar},\quad q(0)=q(\hbar\beta)
\end{gather*}
$$
$S_E$ can be discretized as
$$
\begin{gather*}
    S_E[q]=\int_0^\tau d\tau~\bigg(\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2+V(q(\tau))\bigg)\approx h\sum_i\bigg(\frac{1}{2}m\bigg(\frac{q_{i+1}-q_i}{h}\bigg)^2+V(q_i)\bigg)
\end{gather*}
$$
with $\tau_i=ih$ and $q_i=q(\tau_i)$. $h$ is the imaginary time step. After local update at certain site $j$, $q_j\to q_j+\eta$, the differecne of Euclidean action is
$$
\begin{align*}
    \Delta S_E/h=(S_E[q+\eta]-S_E[q])/h&=\frac{m}{2}\bigg(\bigg(\frac{q_{j+1}-(q_j+\eta)}{h}\bigg)^2 - \bigg(\frac{q_{j+1}-q_j}{h}\bigg)^2 + \bigg(\frac{(q_j+\eta) - q_{j-1}}{h}\bigg)^2 - \bigg(\frac{q_j - q_{j-1}}{h}\bigg)^2\bigg)+V(q_j+\eta)-V(q_j)\\
    &=\frac{m}{2h^2}\bigg((q_{j+1}-q_j-\eta)^2 - (q_{j+1}-q_j)^2 + (q_j+\eta - q_{j-1})^2 - (q_j - q_{j-1})^2\bigg)+V(q_j+\eta)-V(q_j)\\
    &=\frac{m}{2h^2}\bigg(q_{j+1}^2+q_j^2+\eta^2-2q_{j+1}q_j-2q_{j+1}\eta+2q_j\eta-q_{j+1}^2-q_j^2+2q_{j+1}q_j\\
    &\qquad\quad+ q_j^2+q_{j-1}^2+\eta^2+2q_j\eta-2q_jq_{j-1}-2q_{j-1}\eta - q_j^2-q_{j-1}^2+2q_jq_{j-1}\bigg)+V(q_j+\eta)-V(q_j)\\
    \Delta S_E&=\frac{m}{h}(\eta^2-\eta(q_{j+1}-2q_j+q_{j-1}))+h(V(q_j+\eta)-V(q_j))\\
\end{align*}
$$


In [1]:
import numpy as np
import numpy.random as nr
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
def V_sho(x, w=0.1):
    # Simple Harmonic potential
    return 0.5(w**2)*(x**2)


def V_higgs(x, a=3, l=10):
    # Higgs potential
    return l*(x**2 - a**2)**2

In [ ]:
class pathIntegral_particle:
    def __init__(self, N, dt, V):
        self.dt = dt
        self.V = V
        self.N = N

        self.q = nr.randn(N)
        self.beta = self.N*self.dt # inverse temperature


    def S_E(self):
        # Euclidean action or Energy of system

        v = np.roll(self.q, -1) - self.q
        S = np.sum(0.5*v*v/self.dt + self.V(self.q)*self.dt)

        return S
    

    def Metropolis(self, r=0.1):
        # Metropolis altorithm

        j = nr.randint(self.N) # Random index for update
        eta = nr.uniform(-r, r) # Random displacement at q[j]

        # Difference of S_E after local update
        jp = (j+1)%self.N
        jm = (j-1)%self.N
        dS = 1/self.dt*(eta*eta - eta*(self.q[jp] - 2*self.q[j] + self.q[jm])) + self.dt*(self.V(self.q[j]+eta) - self.V(self.q[j]))

        if dS <= 0:
            # Energy decreasing
            self.q[j] += eta

        elif nr.rand() < np.exp(-dS):
            self.q[j] += eta


    def MonteCarlo(self, eq_steps=10**6, MC_steps=10**6):
        #i = 0


        for _ in range(eq_steps):
            # Equilibriate steps
            self.Metropolis()

        for _ in range(MC_steps):
            # Monte Carlo steps
            self.Metropolis()